## Actividad 7

Tipo de datos y objetos
TGAD - Taller de Programación para el análisis de datos


In [1]:
# 1.-Obtenga todas las “PRINCIPALES VARIABLES” de esta página web: 
#    https://www.bcra.gob.ar/

import pandas as pd

# La función read_html lee las tablas HTML de una página y devuelve una lista de DataFrames.
# En este caso, usamos la sección de indicadores del BCRA porque allí aparece la tabla visible
# con las principales variables.
url = "https://www.bcra.gob.ar/estadisticas-indicadores/"

tablas = pd.read_html(url)

# tablas es una lista, por eso usamos el índice [0] para tomar el primer DataFrame encontrado.
tabla_principales = tablas[0]

# astype(str) convierte todas las celdas a texto para poder tratarlas de manera uniforme.
# apply es un método de pandas que aplica una función a cada fila.
# axis=1 indica que la función se aplica fila por fila y no columna por columna.
# lambda crea una función corta sin definirla con def.
# join une los valores de cada fila con el separador " | ".
# strip quita espacios sobrantes y lower ayuda a identificar el texto "nan".
list_rows = tabla_principales.astype(str).apply(
    lambda fila: " | ".join([
        str(valor).strip() 
        for valor in fila 
        if str(valor).lower() != "nan"
    ]),
    axis=1
).tolist()  # tolist convierte el resultado final en una lista de Python.

print("Cantidad de principales variables encontradas:", len(list_rows))
list_rows

Cantidad de principales variables encontradas: 46


['Reservas Internacionales del BCRA (en millones de dólares - cifras provisorias sujetas a cambio de valuación) | 17/04/2026 | 45.793',
 '',
 'Régimen de bandas cambiarias (Límite superior) | 21/04/2026 | 1.688,68',
 'Régimen de bandas cambiarias (Límite inferior) | 21/04/2026 | 82598',
 '',
 'Tipo de Cambio Minorista ($ por USD) Comunicación B 9791 - Promedio vendedor | 20/04/2026 | 1.397,09',
 'Tipo de Cambio Mayorista ($ por USD) Comunicación A 3500 - Referencia | 21/04/2026 | 1.379,0516',
 '',
 'Tasas de interés | Tasas de interés | Tasas de interés',
 'TAMAR en pesos de bancos privados (en % n.a.) | 20/04/2026 | 223125',
 'TAMAR en pesos de bancos privados (en % e.a.) | 20/04/2026 | 247300',
 'BADLAR en pesos de bancos privados (en % n.a.) | 20/04/2026 | 213125',
 'BADLAR en pesos de bancos privados (en % e.a.) | 20/04/2026 | 235100',
 'TM20 en pesos de bancos privados (en % n.a.) | 20/04/2026 | 216250',
 'Tasas de interés por préstamos entre entidades financiera privadas (BAIBAR)

In [2]:
# 2.- Haga toda la manipulación necesaria para obtener un dataframe listo para exportar 
#     como tabla de datos

# Primero armamos un dataframe con el texto crudo que salió en el punto 1.
df_crudo = pd.DataFrame(list_rows, columns=["registro_crudo"])

# Quitamos las filas vacías que aparecieron como separadores visuales en la página.
df_crudo = df_crudo[df_crudo["registro_crudo"] != ""].copy()

# apply recorre cada fila y la transforma en una lista con sus partes.
# En este caso, usamos " | " como separador porque así vienen las variables del BCRA.
partes = df_crudo["registro_crudo"].apply(lambda texto: texto.split(" | "))

# apply vuelve a recorrer cada lista para corregir los casos especiales.
# Si una fila trae más de tres partes, unimos la descripción intermedia en una sola columna.
def normalizar_partes(lista_partes):
    if len(lista_partes) == 3:
        return lista_partes
    if len(lista_partes) > 3:
        variable = lista_partes[0]
        fecha = lista_partes[-2]
        valor = lista_partes[-1]
        descripcion = " | ".join(lista_partes[1:-2])
        if descripcion:
            variable = f"{variable} | {descripcion}"
        return [variable, fecha, valor]
    if len(lista_partes) == 2:
        return [lista_partes[0], "", lista_partes[1]]
    return [lista_partes[0], "", ""]

partes = partes.apply(normalizar_partes)

# Finalmente convertimos la lista de listas en un dataframe limpio y listo para exportar.
df_final = pd.DataFrame(partes.tolist(), columns=["Variable", "Fecha", "Valor"])

df_final

,Variable,Fecha,Valor
0,Reservas Internacionales del BCRA (en millones...,17/04/2026,45.793
1,Régimen de bandas cambiarias (Límite superior),21/04/2026,"1.688,68"
2,Régimen de bandas cambiarias (Límite inferior),21/04/2026,82598
3,Tipo de Cambio Minorista ($ por USD) Comunicac...,20/04/2026,"1.397,09"
4,Tipo de Cambio Mayorista ($ por USD) Comunicac...,21/04/2026,"1.379,0516"
5,Tasas de interés,Tasas de interés,Tasas de interés
6,TAMAR en pesos de bancos privados (en % n.a.),20/04/2026,223125
7,TAMAR en pesos de bancos privados (en % e.a.),20/04/2026,247300
8,BADLAR en pesos de bancos privados (en % n.a.),20/04/2026,213125
9,BADLAR en pesos de bancos privados (en % e.a.),20/04/2026,235100


In [3]:
# 3.- Observe que en dos filas le quedarán dos valores. Resolverlo

# Mostramos las filas que tenían una estructura distinta para confirmar la limpieza.
# El resultado ya quedó normalizado en df_final por el trabajo hecho en el punto 2.
df_final[df_final["Variable"].str.contains("Tasas de interés|Comunicado P 14290|ICL", regex=True)]

,Variable,Fecha,Valor
5,Tasas de interés,Tasas de interés,Tasas de interés
11,Tasas de interés por préstamos entre entidades...,20/04/2026,2129
12,Tasas de interés por depósitos a 30 días de pl...,20/04/2026,2100
17,Tasa de interés para uso de la Justicia - Comu...,21/04/2026,"25.233,6076"
35,Índice para Contratos de Locación (ICL) ley 27...,21/04/2026,3171
